[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anandjha90/Advanced-Python-for-GenAI-Prep-Training/blob/main/notebooks/01_Advanced_Python_for_GenAI.ipynb)

# Module 1: Advanced Python for GenAI (30 min)

The pre-program covered Python basics: variables, data types, operators, control flow, lists, tuples and dictionaries. This module adds the Python features you will meet in every Generative AI code base: comprehensions, flexible functions, decorators, generators, classes, JSON handling with error control, and NumPy/Pandas.

![Figure 1: The learning path in this training. Each module is a building block for GenAI.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d01_roadmap.png)
*Figure 1: The learning path in this training. Each module is a building block for GenAI.*


![Figure 2: Where each Python feature in this module is used inside a typical GenAI (RAG) application.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d02_python_genai.png)
*Figure 2: Where each Python feature in this module is used inside a typical GenAI (RAG) application.*


> **How to run the code**
>
> Open colab.research.google.com, click New notebook, paste each code block into its own cell and press Shift + Enter. Every block in Modules 1 to 5 is self-contained, so it can be run on its own and in any order. All libraries used (NumPy, Pandas, SciPy, Matplotlib, Seaborn, scikit-learn, TensorFlow/Keras) are pre-installed in Colab.

## 1.1 Comprehensions

A comprehension builds a new list, dictionary or set from an existing collection in a single readable line. The pattern is `[expression for item in collection if condition]`. It replaces the "create empty list, loop, append" pattern and is usually faster because the loop runs inside Python's C implementation.

- **List comprehension** `[...]`: transform and/or filter items.
- **Dict comprehension** `{k: v ...}`: build key-value lookups, e.g. product to price.
- **Set comprehension** `{...}`: unique values only.
- **Nested comprehension**: flatten lists of lists, a very common step when working with tokenised text.

> **Real-world example**
>
> An e-commerce team scrapes product titles from partner sites. The titles arrive with extra spaces, random capitalisation, blanks and missing values. Before they are sent to an embedding model they must be cleaned and de-duplicated. One comprehension does the job.

In [ ]:
# Real-world example: cleaning raw product titles scraped from an e-commerce site
raw_titles = ["  Wireless EARBUDS ", "smart watch", "", "  Yoga Mat", None, "AIR fryer  "]

# 1) List comprehension with a filter: keep valid titles and clean them
clean_titles = [t.strip().title() for t in raw_titles if t and t.strip()]
print("Clean titles:", clean_titles)

# 2) Dict comprehension: product -> price after a 10% festival discount
prices = {"Earbuds": 1999, "Smart Watch": 3999, "Yoga Mat": 899}
sale_prices = {name: round(price * 0.9, 2) for name, price in prices.items()}
print("Sale prices:", sale_prices)

# 3) Set comprehension: unique categories (duplicates removed automatically)
orders = [("Earbuds", "Electronics"), ("Yoga Mat", "Sports"), ("Watch", "Electronics")]
categories = {category for _, category in orders}
print("Unique categories:", sorted(categories))

# 4) Conditional expression inside a comprehension: label each order
order_values = [450, 2300, 999, 5600]
labels = ["High" if v > 2000 else "Low" for v in order_values]
print("Order labels:", labels)

# 5) Nested comprehension: flatten tokenised sentences (very common in NLP)
sentences = [["deliver", "fast"], ["great", "quality"], ["return", "item"]]
tokens = [word for sentence in sentences for word in sentence]
print("Flattened tokens:", tokens)

## 1.2 Functions: default arguments, *args, **kwargs and lambda

Functions package logic so it can be reused and tested. Four features make functions flexible:

- **Default arguments** (`role="assistant"`) let callers skip common options.
- **\*args** collects any number of positional values into a tuple, e.g. any number of item prices.
- **\*\*kwargs** collects any number of named options into a dictionary. LLM SDKs use this pattern heavily, for example `client.chat(model=..., temperature=..., max_tokens=...)`.
- **lambda** creates a small one-line function, usually passed to `sorted`, `filter` or `map`.

> **Real-world example**
>
> Every GenAI application builds prompts. A prompt builder function takes the user's question, optional context retrieved from documents, and any number of extra instructions (tone, length, language) and returns one prompt string. The same function serves the chatbot, the email writer and the product-description generator.

In [ ]:
# Real-world example: a reusable prompt builder for an LLM application
def build_prompt(question, context="", role="helpful shopping assistant", **options):
    """Return a prompt string. Extra keyword options become instructions."""
    prompt = f"You are a {role}.\n"
    if context:
        prompt += f"Use only this context:\n{context}\n"
    prompt += f"Question: {question}\n"
    for key, value in options.items():
        prompt += f"Instruction - {key}: {value}\n"
    return prompt

print(build_prompt("What is the return window?",
                   context="Products can be returned within 7 days of delivery.",
                   tone="friendly", max_words=40))

# *args collects any number of positional values into a tuple
def total_bill(*item_prices, gst_rate=0.18):
    subtotal = sum(item_prices)
    return round(subtotal * (1 + gst_rate), 2)

print("Bill with GST:", total_bill(499, 1299, 250))

# lambda + sorted / filter / map on a product list
products = [
    {"name": "Earbuds", "price": 1999, "rating": 4.3},
    {"name": "Yoga Mat", "price": 899, "rating": 4.6},
    {"name": "Air Fryer", "price": 5999, "rating": 4.1},
    {"name": "Green Tea", "price": 349, "rating": 4.4},
]
top_rated = sorted(products, key=lambda p: p["rating"], reverse=True)
under_1000 = list(filter(lambda p: p["price"] < 1000, products))
names_upper = list(map(lambda p: p["name"].upper(), products))

print("Top rated:", [p["name"] for p in top_rated])
print("Under Rs 1000:", [p["name"] for p in under_1000])
print("Upper-case names:", names_upper)

## 1.3 Decorators

A decorator is a function that wraps another function to add behaviour (logging, timing, caching, retrying) without changing the original code. Writing `@timer` above a function is shorthand for `func = timer(func)`. A decorator factory such as `@retry(max_attempts=5)` is a function that returns a decorator, so it can take its own settings. `functools.wraps` keeps the original function's name and docstring.

> **Real-world example**
>
> LLM APIs fail now and then with timeouts or rate-limit errors. Production code wraps every API call with a retry decorator (often with a growing wait time) and a timer to monitor latency. Libraries such as `tenacity` and LangChain's retry logic work exactly this way.

In [ ]:
import time
import random
import functools

def timer(func):
    """Decorator: prints how long a function took."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[timer] {func.__name__} took {time.perf_counter() - start:.3f} s")
        return result
    return wrapper

def retry(max_attempts=3, wait_seconds=0.2):
    """Decorator factory: retries a function when it raises ConnectionError."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as err:
                    print(f"[retry] attempt {attempt} failed: {err}")
                    time.sleep(wait_seconds)
            raise RuntimeError(f"{func.__name__} failed after {max_attempts} attempts")
        return wrapper
    return decorator

random.seed(7)

@timer
@retry(max_attempts=5)
def call_llm_api(prompt):
    """Simulates an unreliable LLM API that fails about half the time."""
    if random.random() < 0.5:
        raise ConnectionError("API timeout")
    return f"LLM answer to: '{prompt}'"

print(call_llm_api("Summarise yesterday's sales in one line"))

## 1.4 Generators

A generator is a function that uses `yield` instead of `return`. It produces values one at a time, only when asked, and remembers where it stopped. This is called lazy evaluation. A list of one million numbers uses about 8 MB; a generator for the same numbers uses about 200 bytes.

- Use generators for large files, streaming API responses and batching.
- A generator expression `(x for x in data)` is the lazy version of a list comprehension.
- When an LLM "streams" its answer word by word, the client library is iterating over a generator.

> **Real-world example**
>
> Retrieval-Augmented Generation (RAG) starts by splitting long documents (return policy, product manuals) into overlapping chunks, because embedding models accept limited text. The overlap keeps sentences that cross a boundary from losing meaning. Chunks are then sent to the embedding API in batches.

In [ ]:
import sys

# Real-world example: splitting a long document into overlapping chunks before
# sending it to an embedding model (this is step 1 of every RAG pipeline)
def chunk_text(text, chunk_size=10, overlap=3):
    """Yield chunks of `chunk_size` words that overlap by `overlap` words."""
    words = text.split()
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        yield " ".join(words[start:start + chunk_size])
        if start + chunk_size >= len(words):
            break

policy = ("Customers can return most products within 7 days of delivery. "
          "Electronics must be unused and in the original packaging. "
          "Refunds are processed to the original payment method within 5 business days. "
          "Cash on delivery orders are refunded to the bank account given by the customer.")

for i, chunk in enumerate(chunk_text(policy), start=1):
    print(f"Chunk {i}: {chunk}")

# Batching: send reviews to an API in groups instead of one huge request
def batch(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

reviews = [f"review_{i}" for i in range(1, 11)]
for b in batch(reviews, 4):
    print("Sending batch:", b)

# Why generators? They produce values lazily and use almost no memory
squares_list = [x * x for x in range(1_000_000)]
squares_gen = (x * x for x in range(1_000_000))
print(f"List size: {sys.getsizeof(squares_list):,} bytes")
print(f"Generator size: {sys.getsizeof(squares_gen):,} bytes")

## 1.5 Classes and dataclasses

Object-oriented programming groups data (attributes) and behaviour (methods) into a class. A `@dataclass` automatically writes the `__init__`, `__repr__` and comparison methods for classes that mainly hold data, so the code stays short. `field(default_factory=list)` gives every object its own empty list.

> **Real-world example**
>
> A chatbot must send the conversation history to the LLM with every request, but models have a context limit (measured in tokens; we use words here for simplicity). A `Conversation` class stores messages, keeps the system instruction, and trims the oldest turns when the budget is exceeded. It also converts messages to the `[{"role": ..., "content": ...}]` format that LLM APIs expect.

In [ ]:
from dataclasses import dataclass, field
from typing import List

# Real-world example: managing chat history for an LLM chatbot with a word budget
@dataclass
class Message:
    role: str        # "system", "user" or "assistant"
    content: str

    def word_count(self):
        return len(self.content.split())

@dataclass
class Conversation:
    max_words: int = 30
    messages: List[Message] = field(default_factory=list)

    def add(self, role, content):
        self.messages.append(Message(role, content))
        self._trim()

    def total_words(self):
        return sum(m.word_count() for m in self.messages)

    def _trim(self):
        # Keep the system message (index 0); drop the oldest turns when over budget
        while self.total_words() > self.max_words and len(self.messages) > 2:
            removed = self.messages.pop(1)
            print(f"  (trimmed old message: '{removed.content[:30]}...')")

    def to_api_format(self):
        return [{"role": m.role, "content": m.content} for m in self.messages]

chat = Conversation(max_words=30)
chat.add("system", "You are a helpful e-commerce support assistant.")
chat.add("user", "Where is my order ORD200145? It was due yesterday.")
chat.add("assistant", "Your order is out for delivery and will arrive today by 7 PM.")
chat.add("user", "Can I change the delivery address to my office in Koramangala?")

print("Words in history:", chat.total_words())
for msg in chat.to_api_format():
    print(msg)

## 1.6 Inheritance and polymorphism

Inheritance lets a child class reuse and extend a parent class. Polymorphism means different classes respond to the same method call in their own way. A base class defines the interface (`generate()`), and each child provides its own implementation. Code that uses the models does not care which model it receives.

> **Real-world example**
>
> Frameworks such as LangChain and LlamaIndex define a base LLM class; OpenAI, Anthropic, Gemini and local models are child classes. You can switch providers by changing one line. Teams also write cheap fake models (like `EchoLLM` below) to test their pipelines without paying for API calls.

In [ ]:
# Real-world example: the "common interface, many models" pattern used by
# LangChain / LlamaIndex. Every model class exposes the same generate() method.
class BaseLLM:
    def __init__(self, name, temperature=0.0):
        self.name = name
        self.temperature = temperature

    def generate(self, prompt):
        raise NotImplementedError("Child classes must implement generate()")

    def __repr__(self):
        return f"{self.__class__.__name__}(name='{self.name}', temperature={self.temperature})"

class EchoLLM(BaseLLM):
    """A fake model for testing pipelines without paying for API calls."""
    def generate(self, prompt):
        return f"[{self.name}] You said: {prompt}"

class FAQBot(BaseLLM):
    """A rule-based 'model' that answers from a small knowledge base."""
    def __init__(self, name, faq):
        super().__init__(name)
        self.faq = faq

    def generate(self, prompt):
        for keyword, answer in self.faq.items():
            if keyword in prompt.lower():
                return f"[{self.name}] {answer}"
        return f"[{self.name}] Sorry, I will connect you to a human agent."

faq = {"refund": "Refunds reach your account in 5 business days.",
       "cancel": "You can cancel before the order is shipped."}

models = [EchoLLM("echo-v1"), FAQBot("faq-bot", faq)]
for model in models:                       # polymorphism: same call, different behaviour
    print(model)
    print("  ->", model.generate("How long does a refund take?"))

## 1.7 JSON and exception handling

JSON is the standard format for data exchanged with APIs. `json.loads` converts a JSON string to a Python dictionary and `json.dumps`/`json.dump` does the reverse. Exception handling keeps a program running when something goes wrong:

- `try`: code that might fail.
- `except`: what to do if a specific error happens.
- `else`: runs only if no error happened.
- `finally`: always runs (closing files, releasing connections).

> **Real-world example**
>
> We ask an LLM to return sentiment as JSON. Sometimes it returns clean JSON, sometimes it adds friendly text around the JSON, sometimes it returns plain text or broken JSON. A robust parser tries strict parsing first, then extracts the `{...}` block with a regular expression, and finally falls back to a safe default instead of crashing the pipeline.

In [ ]:
import json
import re

# Real-world example: LLMs are asked to return JSON, but they do not always obey.
llm_outputs = [
    '{"sentiment": "positive", "score": 0.92}',
    'Sure! Here is the result:\n{"sentiment": "negative", "score": 0.15}\nHope this helps.',
    'I think the sentiment is neutral.',
    '{"sentiment": "positive", "score": 0.8',          # broken JSON (missing brace)
]

def parse_llm_json(text):
    """Try strict JSON first, then extract a {...} block, else return a safe default."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return {"sentiment": "unknown", "score": None}

results = []
for raw in llm_outputs:
    parsed = parse_llm_json(raw)
    results.append(parsed)
    print(f"{raw[:35]!r:40} -> {parsed}")

# try / except / else / finally
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero - returning 0")
        result = 0
    else:
        print("Division successful")
    finally:
        print("Done")
    return result

print("Conversion rate:", safe_divide(45, 900))
print("Conversion rate:", safe_divide(45, 0))

# Save and reload results as a JSON file
with open("llm_results.json", "w") as f:
    json.dump(results, f, indent=2)
with open("llm_results.json") as f:
    print("Reloaded from file:", json.load(f))

## 1.8 NumPy and Pandas essentials

**NumPy** stores numbers in fixed-type arrays and applies operations to the whole array at once (vectorisation). This is 50 to 100 times faster than a Python loop and is the foundation of every ML and DL library. **Pandas** adds labelled tables (DataFrames) with SQL-like operations: filter, group by, aggregate, pivot and join.

| SQL | Pandas |
|---|---|
| SELECT col1, col2 | df[["col1", "col2"]] |
| WHERE revenue > 8000 | df[df["revenue"] > 8000] |
| GROUP BY category + SUM | df.groupby("category")["revenue"].sum() |
| JOIN targets ON category | df.merge(targets, on="category") |
| PIVOT | df.pivot_table(index=..., columns=..., values=...) |


> **Real-world example**
>
> A category manager wants revenue by category and city, compared with monthly targets. In Pandas this is a group-by, a pivot table and a merge, the same steps as in SQL.

In [ ]:
import time
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# 1) Vectorisation: NumPy is much faster than a Python loop
prices = np.random.default_rng(0).uniform(100, 5000, 1_000_000)

start = time.perf_counter()
loop_result = [p * 0.9 for p in prices]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vector_result = prices * 0.9
vector_time = time.perf_counter() - start
print(f"Loop: {loop_time:.3f} s | NumPy: {vector_time:.4f} s | "
      f"NumPy is ~{loop_time / vector_time:.0f}x faster")

# 2) Pandas: a small order table
rng = np.random.default_rng(42)
n = 1000
orders = pd.DataFrame({
    "order_id": [f"ORD{i}" for i in range(1, n + 1)],
    "category": rng.choice(["Electronics", "Fashion", "Books", "Grocery"], n),
    "city": rng.choice(["Bengaluru", "Mumbai", "Delhi"], n),
    "price": rng.integers(200, 5000, n),
    "quantity": rng.integers(1, 4, n),
})
orders["revenue"] = orders["price"] * orders["quantity"]       # vectorised new column
print(orders.head())

# Filter, group, aggregate
high_value = orders[orders["revenue"] > 8000]
print("\nHigh-value orders:", len(high_value))

summary = (orders.groupby("category")
                 .agg(orders=("order_id", "count"),
                      total_revenue=("revenue", "sum"),
                      avg_revenue=("revenue", "mean"))
                 .sort_values("total_revenue", ascending=False)
                 .round(0))
print("\nRevenue by category:\n", summary)

# Pivot table: city x category revenue
pivot = orders.pivot_table(index="city", columns="category", values="revenue", aggfunc="sum")
print("\nCity x Category revenue:\n", pivot)

# Merge with a targets table (like a SQL JOIN)
targets = pd.DataFrame({"category": ["Electronics", "Fashion", "Books", "Grocery"],
                        "target": [2_000_000, 1_500_000, 900_000, 800_000]})
result = summary.reset_index().merge(targets, on="category")
result["achieved_pct"] = (result["total_revenue"] / result["target"] * 100).round(1)
print("\nTarget achievement:\n", result[["category", "total_revenue", "target", "achieved_pct"]])

## 1.9 Module 1 practical: review-processing pipeline

This practical combines everything from Module 1. Raw customer reviews are cleaned with regular expressions, scored with a simple keyword sentiment function, streamed through a generator, stored as dataclass objects, summarised with Pandas and exported to JSON. In a real GenAI system, the exported negative reviews would be sent to an LLM to summarise root causes.

In [ ]:
import json
import re
from dataclasses import dataclass, asdict
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# PRACTICAL: a mini review-processing pipeline (the kind of pre-processing done
# before reviews are summarised by an LLM)
raw_reviews = [
    ("ORD1", "Electronics", "  GREAT earbuds!!! Sound quality is excellent  "),
    ("ORD2", "Fashion", "Size did not fit. Very disappointed, returning it."),
    ("ORD3", "Fashion", "Nice fabric and fast delivery :)"),
    ("ORD4", "Electronics", "Stopped working after 2 days. Poor quality."),
    ("ORD5", "Books", ""),
    ("ORD6", "Grocery", "Fresh and well packed, good value."),
    ("ORD7", "Electronics", "Okay product, average battery life."),
]

POSITIVE = {"great", "excellent", "nice", "fast", "fresh", "good", "love"}
NEGATIVE = {"poor", "disappointed", "returning", "stopped", "bad", "damaged", "late"}

@dataclass
class Review:
    order_id: str
    category: str
    text: str
    sentiment: str = "neutral"
    score: int = 0

def clean(text):
    text = text.lower().strip()
    return re.sub(r"[^a-z0-9\s]", "", text)        # remove punctuation and emojis

def score_sentiment(text):
    words = set(text.split())
    score = len(words & POSITIVE) - len(words & NEGATIVE)
    label = "positive" if score > 0 else "negative" if score < 0 else "neutral"
    return label, score

def review_stream(rows):
    """Generator: yields cleaned Review objects and skips empty reviews."""
    for order_id, category, text in rows:
        cleaned = clean(text)
        if not cleaned:
            continue
        label, score = score_sentiment(cleaned)
        yield Review(order_id, category, cleaned, label, score)

reviews = list(review_stream(raw_reviews))
df = pd.DataFrame([asdict(r) for r in reviews])
print(df.to_string())

print("\nSentiment by category:")
print(pd.crosstab(df["category"], df["sentiment"]))

# Export negative reviews as JSON - ready to send to an LLM for root-cause summary
negatives = df[df["sentiment"] == "negative"].to_dict(orient="records")
with open("negative_reviews.json", "w") as f:
    json.dump(negatives, f, indent=2)
print(f"\nSaved {len(negatives)} negative reviews to negative_reviews.json")

> **Discussion**
>
> The keyword approach gives "Okay product, average battery life" a neutral score, which is right, but it would fail on "not great". This weakness is why we move to machine learning (Module 4) and deep learning (Module 5), where models learn sentiment from data instead of fixed word lists.